# Table 1 Analytic Summary (built on your own dataset loaders)

This notebook reuses your existing loader functions as they are, then computes for each of the 18 configurations the analytic sample size, the feature count after One Hot Encoding, and the sensitive group sizes, matching what Table 1 needs.

Run all cells in order on your own machine, since the file paths inside the loaders are your local paths.

In [2]:
import pandas as pd
import numpy as np
import io
import requests
from scipy.io import arff

import warnings
warnings.filterwarnings('ignore')
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    fetch_ucirepo = None


## Section 1, your original 5 loaders (Adult, COMPAS, German, Credit, Law), unchanged

In [4]:
def load_adult():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    adult = pd.read_csv(r"../data/adult/adult.data", names=columns, header=None)
    adult = adult.replace(' ?', np.nan).dropna()
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    return adult, adult_features, 'sex_binary'


def load_compas():
    compas = pd.read_csv(r"../data/compas/compas-scores.csv")
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    return compas, compas_features, 'sex_binary'


def load_german():
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    german = pd.read_csv(r"../data/german/german.data", sep=' ', names=german_columns, header=None)
    def extract_sex(s):
        return 'male' if s in ['A91', 'A93', 'A94'] else ('female' if s in ['A92', 'A95'] else 'unknown')
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    return german, german_features, 'sex_binary'


def load_credit():
    credit = pd.read_csv(r"../data/default of credit card clients/default of credit card clients.csv", header=0)
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default'
        ]
    credit = credit.drop(columns=['ID'])
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    return credit, credit_features, 'sex_binary'


def load_law():
    lsac = pd.read_csv(r"../data/law/law_dataset.csv")
    lsac['sex'] = lsac['male'].map({1: 'Male', 0: 'Female', 1.0: 'Male', 0.0: 'Female'})
    lsac = lsac[lsac['sex'].notna()]
    lsac['sex_binary'] = (lsac['sex'] == 'Male').astype(int)
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary', 'racetxt'] 
    lsac_features = [c for c in lsac.columns if c not in drop_cols]
    return lsac, lsac_features, 'sex_binary'


## Section 1b, race and combined variants of Adult, COMPAS, Law

In [6]:
def load_adult_race():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv(r"../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['race_binary'] = (df['race'] == 'White').astype(int)
    features = [c for c in df.columns if c not in ["income", "sex", "race", "race_binary"]]
    return df, features, 'race_binary'

def load_adult_combined():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv(r"../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['sex_binary']  = (df['sex'].str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'White').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["income","sex","race","sex_binary","race_binary","combined"]]
    return df, features, 'combined'

def load_compas_race():
    df = pd.read_csv(r"../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    features = [c for c in df.columns if c not in ["is_recid","sex","race","race_binary"]]
    return df, features, 'race_binary'

def load_compas_combined():
    df = pd.read_csv(r"../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["is_recid","sex","race","sex_binary","race_binary","combined"]]
    return df, features, 'combined'

def load_law_race():
    df = pd.read_csv(r"../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['race_binary'] = df['racetxt'].astype(int)
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','race_binary']]
    return df, features, 'race_binary'

def load_law_combined():
    df = pd.read_csv(r"../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = df['racetxt'].astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','sex_binary','race_binary','combined']]
    return df, features, 'combined'


## Section 2, new loaders, Diabetes 130-US, Dutch Census, MEPS

In [8]:
import os
def load_diabetes(mode='gender'):
    ds = fetch_ucirepo(id=296)
    df = ds.data.features.copy()

    drop_cols = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[df['gender'].isin(['Male', 'Female'])]
    df = df[df['race'] != '?'].dropna(subset=['gender', 'race'])

    df['sex_binary'] = (df['gender'] == 'Male').astype(int)

    df_race = df[df['race'].isin(['Caucasian', 'AfricanAmerican'])].copy()
    df_race['race_binary'] = (df_race['race'] == 'Caucasian').astype(int)
    df_race['combined'] = df_race['race_binary'] * 2 + df_race['sex_binary']

    base_excl = ['gender', 'race', 'sex_binary', 'race_binary', 'combined']

    if mode == 'gender':
        features = [c for c in df.columns if c not in base_excl]
        return df, features, 'sex_binary'
    elif mode == 'race':
        features = [c for c in df_race.columns if c not in base_excl]
        return df_race, features, 'race_binary'
    elif mode == 'combined':
        features = [c for c in df_race.columns if c not in base_excl]
        return df_race, features, 'combined'
    else:
        raise ValueError(f"mode must be 'gender'/'race'/'combined', got '{mode}'")


def load_dutch(mode='gender'):
    if mode != 'gender':
        raise ValueError("Dutch Census only supports mode='gender' (no race attribute)")

    url = ('https://raw.githubusercontent.com/tailequy/fairness_dataset'
           '/main/Dutch_census/dutch_census_2001.arff')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data, meta = arff.loadarff(io.StringIO(resp.text))
        df = pd.DataFrame(data)
        for col in df.select_dtypes(['object']).columns:
            df[col] = df[col].str.decode('utf-8')
    except Exception:
        df = pd.read_csv('./data/dutch/dutch_census_2001.csv')

    sex_col = [c for c in df.columns if c.lower() == 'sex'][0]
    unique_vals = set(df[sex_col].astype(str).unique())
    if unique_vals.issubset({'1', '2'}):
        df['sex_binary'] = (df[sex_col].astype(str) == '1').astype(int)
    else:
        df['sex_binary'] = (df[sex_col].str.lower().isin(['male', 'm', '1'])).astype(int)

    target_col = next((c for c in df.columns if c.lower() in ['occupation', 'label', 'class']), None)
    excl = [sex_col, 'sex_binary'] + ([target_col] if target_col else [])
    features = [c for c in df.columns if c not in excl]
    return df, features, 'sex_binary'


def load_meps(mode='gender'):
    import aif360
    meps_dir = os.path.join(os.path.dirname(aif360.__file__), 'data', 'raw', 'meps')
    df = pd.read_csv(meps_dir + r'/h192.csv', low_memory=False)

    keep_cols = ['AGE31X','AGE42X','AGE53X','SEX','RACEV1X','MARRY31X','MARRY42X','MARRY53X',
                 'EDUCYR','HIDEG','FTSTU31X','FTSTU42X','FTSTU53X','ACTDTY31','ACTDTY42','ACTDTY53',
                 'HONRDC31','HONRDC42','HONRDC53','WRGFD31X','WRGFD42X','WRGFD53X',
                 'TTLP16X','POVCAT16','INSCOV16']
    available = [c for c in keep_cols if c in df.columns]
    df = df[available].dropna()

    df['sex_binary']  = (df['SEX'] == 1).astype(int)
    df['race_binary'] = (df['RACEV1X'] == 1).astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']

    base_excl = ['SEX', 'RACEV1X', 'sex_binary', 'race_binary', 'combined']
    features = [c for c in df.columns if c not in base_excl]

    if mode == 'gender':
        return df, features, 'sex_binary'
    elif mode == 'race':
        return df, features, 'race_binary'
    elif mode == 'combined':
        return df, features, 'combined'
    else:
        raise ValueError(f"mode must be gender/race/combined, got '{mode}'")


## Driver, maps each of the 18 configurations to the matching loader call

In [10]:
CONFIG_LOADERS = {
    "D1_gender": lambda: load_adult(),
    "D1_race":   lambda: load_adult_race(),
    "D1_comb":   lambda: load_adult_combined(),
    "D2_gender": lambda: load_compas(),
    "D2_race":   lambda: load_compas_race(),
    "D2_comb":   lambda: load_compas_combined(),
    "D3_gender": lambda: load_german(),
    "D4_gender": lambda: load_credit(),
    "D5_gender": lambda: load_law(),
    "D5_race":   lambda: load_law_race(),
    "D5_comb":   lambda: load_law_combined(),
    "D6_gender": lambda: load_diabetes(mode='gender'),
    "D6_race":   lambda: load_diabetes(mode='race'),
    "D6_comb":   lambda: load_diabetes(mode='combined'),
    "D7_gender": lambda: load_dutch(mode='gender'),
    "D8_gender": lambda: load_meps(mode='gender'),
    "D8_race":   lambda: load_meps(mode='race'),
    "D8_comb":   lambda: load_meps(mode='combined'),
}

BASE_DATASET_NAME = {
    "D1": "Adult Income", "D2": "COMPAS", "D3": "German Credit",
    "D4": "Credit Default", "D5": "Law School", "D6": "Diabetes 130 US",
    "D7": "Dutch Census 2001", "D8": "MEPS 20",
}

SENSITIVE_LABEL = {"gender": "Gender", "race": "Race", "comb": "Gender and Race combined"}


In [11]:
def summarize_config(config_name, max_unique_ratio=0.5, numeric_convert_threshold=0.95,
                       datetime_convert_threshold=0.95):
    import re
    loader = CONFIG_LOADERS[config_name]
    df, features, sensitive_col = loader()

    base = config_name.split("_")[0]
    attr = config_name.split("_", 1)[1]

    n_samples = df.shape[0]

    # One Hot Encoding on the feature set actually used for clustering,
    # matching Section 3.3.2, categorical columns expanded, numeric columns
    # kept as they are.
    # Before encoding, recover object columns that are actually numeric
    # (mistyped as string), exclude known identifier style fields by column
    # name (name, first, last, dob), exclude columns that are almost
    # entirely parseable as dates (not nominal categories), and exclude
    # remaining high cardinality object columns that are almost certainly
    # id, case number, or free text fields. This mirrors the fixed
    # preprocess_dataset used in the experiment notebooks, so Table 1
    # reports the same feature counts the clustering runs use.
    feature_df = df[features].copy()
    id_name_pattern = re.compile(r'(name|first|last|dob)', re.IGNORECASE)
    excluded_cols = []
    keep_cols = []

    for col in feature_df.columns:
        if feature_df[col].dtype != object:
            keep_cols.append(col)
            continue

        converted_num = pd.to_numeric(feature_df[col], errors='coerce')
        if converted_num.notna().mean() >= numeric_convert_threshold:
            feature_df[col] = converted_num
            keep_cols.append(col)
            continue

        if id_name_pattern.search(str(col)):
            excluded_cols.append(col)
            continue

        converted_dt = pd.to_datetime(feature_df[col], errors='coerce')
        if converted_dt.notna().mean() >= datetime_convert_threshold:
            excluded_cols.append(col)
            continue

        unique_ratio = feature_df[col].nunique() / n_samples
        if unique_ratio > max_unique_ratio:
            excluded_cols.append(col)
            continue

        keep_cols.append(col)

    if excluded_cols:
        print(f"  {config_name}: excluding likely id, name, free text, or date columns before One Hot Encoding: {excluded_cols}")
        feature_df = feature_df[keep_cols]


    cat_cols = feature_df.select_dtypes(include='object').columns
    feature_df[cat_cols] = feature_df[cat_cols].astype(str)
    
    encoded = pd.get_dummies(feature_df, drop_first=False)
    n_features_ohe = encoded.shape[1]

    groups, counts = np.unique(df[sensitive_col].values, return_counts=True)
    group_sizes_str = " over ".join(str(c) for c in counts)

    return {
        "Configuration": config_name,
        "Base Dataset": BASE_DATASET_NAME.get(base, base),
        "Analytic N": n_samples,
        "Features after One Hot Encoding": n_features_ohe,
        "Sensitive Attribute": SENSITIVE_LABEL.get(attr, attr),
        "Group Sizes": group_sizes_str,
        "Number of Groups": len(groups),
        "Excluded Columns": ", ".join(excluded_cols) if excluded_cols else "",
    }


In [12]:
rows = []
for config_name in CONFIG_LOADERS:
    print("Processing", config_name)
    rows.append(summarize_config(config_name))

table1 = pd.DataFrame(rows)
table1


Processing D1_gender
Processing D1_race
Processing D1_comb
Processing D2_gender
  D2_gender: excluding likely id, name, free text, or date columns before One Hot Encoding: ['name', 'first', 'last', 'compas_screening_date', 'dob', 'c_jail_in', 'c_jail_out', 'c_case_number', 'v_screening_date', 'screening_date']
Processing D2_race
  D2_race: excluding likely id, name, free text, or date columns before One Hot Encoding: ['name', 'first', 'last', 'compas_screening_date', 'dob', 'c_jail_in', 'c_jail_out', 'c_case_number', 'v_screening_date', 'screening_date']
Processing D2_comb
  D2_comb: excluding likely id, name, free text, or date columns before One Hot Encoding: ['name', 'first', 'last', 'compas_screening_date', 'dob', 'c_jail_in', 'c_jail_out', 'c_case_number', 'v_screening_date', 'screening_date']
Processing D3_gender
Processing D4_gender
Processing D5_gender
Processing D5_race
Processing D5_comb
Processing D6_gender
Processing D6_race
Processing D6_comb
Processing D7_gender
Processin

,Configuration,Base Dataset,Analytic N,Features after One Hot Encoding,Sensitive Attribute,Group Sizes,Number of Groups,Excluded Columns
0,D1_gender,Adult Income,30162,97,Gender,9782 over 20380,2,
1,D1_race,Adult Income,30162,97,Race,4229 over 25933,2,
2,D1_comb,Adult Income,30162,97,Gender and Race combined,1887 over 2342 over 7895 over 18038,4,
3,D2_gender,COMPAS,9395,23,Gender,1933 over 7462,2,"name, first, last, compas_screening_date, dob,..."
4,D2_race,COMPAS,7931,23,Race,3256 over 4675,2,"name, first, last, compas_screening_date, dob,..."
5,D2_comb,COMPAS,7931,23,Gender and Race combined,790 over 2466 over 887 over 3788,4,"name, first, last, compas_screening_date, dob,..."
6,D3_gender,German Credit,1000,61,Gender,310 over 690,2,
7,D4_gender,Credit Default,30000,22,Gender,18112 over 11888,2,
8,D5_gender,Law School,18692,9,Gender,8142 over 10550,2,
9,D5_race,Law School,18692,9,Race,1201 over 17491,2,


In [13]:
table1.to_csv("table1_analytic_summary.csv", index=False)
print("Saved to table1_analytic_summary.csv")


Saved to table1_analytic_summary.csv


In [14]:
import pandas as pd
from pathlib import Path

folder   = Path('.')     
out_file = folder / 'all_results.xlsx'

csv_files = sorted(folder.glob('*.csv'))
used = set()
with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
    for f in csv_files:
        df = pd.read_csv(f)
        name = f.stem[:31]     
        i = 1
        while name in used:           
            name = f"{f.stem[:28]}_{i}"; i += 1
        used.add(name)
        df.to_excel(writer, sheet_name=name, index=False)
        print(f"✓ {f.name} -> sheet '{name}' ({df.shape[0]} rows)")

print(f"\n✓ Saved: {out_file}  ({len(csv_files)} sheets)")

✓ k_sensitivity_best_k.csv -> sheet 'k_sensitivity_best_k' (18 rows)
✓ k_sensitivity_summary.csv -> sheet 'k_sensitivity_summary' (126 rows)
✓ optimal_k_summary_partA.csv -> sheet 'optimal_k_summary_partA' (11 rows)
✓ optimal_k_summary_PartB.csv -> sheet 'optimal_k_summary_PartB' (7 rows)
✓ rank_by_metric.csv -> sheet 'rank_by_metric' (8 rows)
✓ rank_table_all.csv -> sheet 'rank_table_all' (288 rows)
✓ rank_table_mfr_no_rawlsian.csv -> sheet 'rank_table_mfr_no_rawlsian' (288 rows)
✓ S2A_majority_vote_k_per_algo_ccf.csv -> sheet 'S2A_majority_vote_k_per_algo_cc' (44 rows)
✓ S2A_majority_vote_k_per_algo_kmf.csv -> sheet 'S2A_majority_vote_k_per_algo_km' (33 rows)
✓ S2A_results_ccf_30runs-1_runtime.csv -> sheet 'S2A_results_ccf_30runs-1_runtim' (44 rows)
✓ S2A_results_ccf_30runs-1_summary.csv -> sheet 'S2A_results_ccf_30runs-1_summar' (44 rows)
✓ S2A_results_kmf_30runs-1_runtime.csv -> sheet 'S2A_results_kmf_30runs-1_runtim' (33 rows)
✓ S2A_results_kmf_30runs-1_summary.csv -> sheet 'S2A_r